In [0]:
%sh
set -e

echo "Installing Colima and docker-compose on Ubuntu..."

# Update package list
apt-get update

# Install dependencies
apt-get install -y curl wget qemu-system-x86 qemu-utils

# Install Docker CLI
apt-get install -y docker.io

# Install Lima (required for Colima)
LIMA_VERSION=$(curl -s https://api.github.com/repos/lima-vm/lima/releases/latest | grep -oP '"tag_name": "\K(.*)(?=")')
curl -LO "https://github.com/lima-vm/lima/releases/download/${LIMA_VERSION}/lima-${LIMA_VERSION:1}-$(uname -s)-$(uname -m).tar.gz"
tar -xzf lima-${LIMA_VERSION:1}-$(uname -s)-$(uname -m).tar.gz
install -d /usr/local/bin
install bin/* /usr/local/bin/
install -d /usr/local/share/lima
cp -r share/lima/* /usr/local/share/lima/
rm -rf lima-${LIMA_VERSION:1}-$(uname -s)-$(uname -m).tar.gz bin share

# Install Colima
COLIMA_VERSION=$(curl -s https://api.github.com/repos/abiosoft/colima/releases/latest | grep -oP '"tag_name": "\K(.*)(?=")')
curl -LO "https://github.com/abiosoft/colima/releases/download/${COLIMA_VERSION}/colima-$(uname -s)-$(uname -m)"
install colima-$(uname -s)-$(uname -m) /usr/local/bin/colima
rm colima-$(uname -s)-$(uname -m)

# Install docker-compose
apt-get install -y docker-compose

echo "Installation complete!"
echo ""
echo "IMPORTANT: Colima cannot be run as root user."
echo "Switch to a non-root user and run: colima start"
echo ""
echo "Example:"
echo "  su - username"
echo "  colima start"

In [0]:
%sh
set -e

echo "Current users (UID >= 1000):"
echo "----------------------------"
awk -F: '$3 >= 1000 && $1 != "nobody" {print $1 " (UID: " $3 ")"}' /etc/passwd
echo ""

# Create colima user
USERNAME="colima"

if id "$USERNAME" &>/dev/null; then
    echo "User '$USERNAME' already exists"
else
    echo "Creating user '$USERNAME'..."
    useradd -m -s /bin/bash "$USERNAME"
    echo "User '$USERNAME' created successfully"
fi

# Add user to docker group if it exists
if getent group docker &>/dev/null; then
    usermod -aG docker "$USERNAME"
    echo "Added '$USERNAME' to docker group"
fi

# Add user to kvm and libvirt groups for virtualization
if getent group kvm &>/dev/null; then
    usermod -aG kvm "$USERNAME"
    echo "Added '$USERNAME' to kvm group"
fi

if getent group libvirt &>/dev/null; then
    usermod -aG libvirt "$USERNAME"
    echo "Added '$USERNAME' to libvirt group"
fi

echo ""
echo "To switch to the colima user, run:"
echo "  su - $USERNAME"
echo ""
echo "Then start Colima with:"
echo "  colima start"

In [0]:
%sh
su - colima
colima start --memory 6

In [0]:
%sh
set -e

cp -r ../solace-single-docker-compose /tmp/solace-single-docker-compose

su - colima
cd /tmp/solace-single-docker-compose/template
ls

echo "Starting Solace PubSub+ with docker-compose..."

# Run docker-compose
docker-compose -f PubSubStandard_singleNode.yml up -d

echo "Solace PubSub+ is starting..."
echo "Web UI will be available at: http://localhost:8080"
echo "Username: admin, Password: admin"
echo ""
echo "Check status with: docker-compose ps"
echo "View logs with: docker-compose logs -f"
echo "Stop with: docker-compose down"

In [ ]:
%sh
su - colima
cd /tmp/solace-single-docker-compose/template
ls
docker-compose -f PubSubStandard_singleNode.yml ps

In [ ]:
%sh
su - colima
cd /tmp/solace-single-docker-compose

docker-compose down